# Train Cross-Encoder NLI tren Kaggle

Notebook nay train Cross-Encoder 3 lop NLI tren AllNLI `pair-class`, sau do danh gia them binary semantic similarity va retrieval reranking cho web similarity search.

## 1. Cai thu vien

In [ ]:
%pip install -q -U "datasets>=3.0" "transformers>=4.40" "accelerate>=1.0" "evaluate>=0.4" "scikit-learn>=1.3" "pandas>=2.0" "pyarrow>=15.0" "huggingface_hub>=0.23"

## 2. Tai source code tu GitHub

In [ ]:
# Dien URL repo GitHub cua nhom vao day truoc khi chay tren Kaggle.
GITHUB_REPOSITORY_URL = "https://github.com/PhDQuang/similarity_search.git"  # vi du: "https://github.com/<user>/<repo>.git"

if GITHUB_REPOSITORY_URL:
    !rm -rf /kaggle/working/similarity_search
    !git clone {GITHUB_REPOSITORY_URL} /kaggle/working/similarity_search
    %cd /kaggle/working/similarity_search
else:
    print("Hay dien GITHUB_REPOSITORY_URL, hoac upload repo source thanh Kaggle Dataset roi cd vao thu muc do.")

## 3. Cai package local cua project

In [ ]:
!python -m pip install -q -e .
!python -m similarity_search.models.train_cross_encoder --help

## 4. Kiem tra GPU

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 5. Smoke test nhanh

Chay truoc voi it mau de kiem tra pipeline. Neu thanh cong moi chay full training.

In [ ]:
!python -m similarity_search.models.train_cross_encoder \
  --output-dir /kaggle/working/cross-encoder-smoke \
  --result-dir /kaggle/working/cross-encoder-smoke-outputs \
  --max-train-samples 2000 \
  --max-dev-samples 500 \
  --max-test-samples 500 \
  --num-train-epochs 0.05 \
  --batch-size 16 \
  --eval-batch-size 32 \
  --eval-steps 20 \
  --save-steps 20 \
  --logging-steps 10 \
  --rerank-queries 50 \
  --rerank-pool-size 10 \
  --skip-benchmark

## 6. Full training Cross-Encoder NLI

Cau hinh mac dinh dung `distilbert-base-uncased`, train 300k mau AllNLI `pair-class`, 1 epoch. Neu bi OOM, giam `--batch-size 16` va tang `--gradient-accumulation-steps 2`.

In [ ]:
!python -m similarity_search.models.train_cross_encoder \
  --train-dataset-name sentence-transformers/all-nli \
  --train-dataset-config pair-class \
  --benchmark-dataset-name phdquang/allnli-pair-class-processed \
  --base-model distilbert-base-uncased \
  --output-dir /kaggle/working/allnli-cross-encoder-nli \
  --result-dir /kaggle/working/cross_encoder_outputs \
  --max-train-samples 300000 \
  --max-dev-samples 20000 \
  --max-test-samples 20000 \
  --num-train-epochs 1 \
  --batch-size 32 \
  --eval-batch-size 64 \
  --gradient-accumulation-steps 1 \
  --learning-rate 2e-5 \
  --warmup-ratio 0.1 \
  --weight-decay 0.01 \
  --max-length 128 \
  --eval-steps 1000 \
  --save-steps 1000 \
  --logging-steps 100 \
  --rerank-queries 1000 \
  --rerank-pool-size 20

## 7. Push len Hugging Face Hub neu can

Neu muon push model, tao Kaggle Secret `HF_TOKEN`, bo comment cell duoi va sua `--hub-model-id`.

In [ ]:
# from kaggle_secrets import UserSecretsClient
# import os
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
#
# !python -m similarity_search.models.train_cross_encoder \
#   --train-dataset-name sentence-transformers/all-nli \
#   --train-dataset-config pair-class \
#   --base-model distilbert-base-uncased \
#   --output-dir /kaggle/working/allnli-cross-encoder-nli-hub \
#   --result-dir /kaggle/working/cross_encoder_outputs_hub \
#   --max-train-samples 300000 \
#   --num-train-epochs 1 \
#   --batch-size 32 \
#   --eval-batch-size 64 \
#   --learning-rate 2e-5 \
#   --push-to-hub \
#   --hub-model-id <username-or-team>/allnli-distilbert-cross-encoder-nli \
#   --hub-private-repo

## 8. File ket qua can download

Sau khi Save Version tren Kaggle, tai cac file ZIP nay o tab Output:

- `/kaggle/working/allnli-cross-encoder-nli.zip`
- `/kaggle/working/cross_encoder_outputs.zip`

Trong `cross_encoder_outputs.zip` co metadata, classification report, confusion matrix, predictions va retrieval rerank predictions.